# Phase 3b -- Dual Fusion: Image + Clinical Metadata (No Text)
## PneumoFusionNet | MIMIC-CXR PA | Image × Clinical Tabular Fusion

### Purpose
This notebook creates a **baseline dual-fusion model** using **only two modalities**:
1. **Chest X-Ray Image** → DenseNet-121 + CBAM (frozen Phase 1 weights) → 1024-d
2. **Clinical Metadata** → MLP (16 features → 64-d)

**No text reports used.** This lets us directly compare:
- Phase 1 (Image only): AUC = 0.820
- **Phase 3b (Image + Clinical): AUC = ?** ← This model
- Phase 3 Full (Image + Text + Clinical): AUC = 0.984

### Architecture
```
Image Encoder (Frozen DenseNet+CBAM, 1024-d) ──┐
                                                  ├── Concat (1088-d) → MLP → Prediction
Metadata Encoder (MLP: 16 → 128 → 64-d) ────────┘
```

### Clinical Features (16 total)
- Demographics: age, gender_M, is_deceased
- Vitals: heart_rate, respiratory_rate, spo2, systolic_bp, diastolic_bp, temperature_f
- Labs: wbc, hemoglobin, hematocrit, creatinine, crp, alk_phos, albumin

In [ ]:
# --- CELL 0: Imports & Reproducibility ---
import os, random, warnings, json, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import torchxrayvision as xrv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, f1_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# --- CELL 1: Configuration & Paths ---
BASE_DIR     = r'c:\\2026\\PneumoFusionNet\\mimic\\1000_dataset'

# Input data
TEXT_CSV     = os.path.join(BASE_DIR, 'phase2_reports_no_impression.csv')   # image paths + labels
CLINICAL_CSV = os.path.join(BASE_DIR, 'phase3_clinical_data.csv')
BBOX_CSV     = os.path.join(BASE_DIR, 'outputs', 'Phase_1.1v2_PA_enhanced', 'lung_bboxes.csv')

# Phase 1 image checkpoint (same as used by Phase 2 & 3)
P1_CKPT      = os.path.join(BASE_DIR, 'outputs', 'Phase_1.1v4_PA_crossval', 'best_model_fold2.pth')

# Output
SAVE_DIR     = os.path.join(BASE_DIR, 'outputs', 'Phase_3_image_clinical')
os.makedirs(SAVE_DIR, exist_ok=True)

# Model
IMG_SIZE       = 224
IMG_FEAT_DIM   = 1024
META_HIDDEN    = 128
META_OUT_DIM   = 64

# Training
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
BATCH_SIZE   = 32          # larger batch since no BERT in memory
LR_FUSION    = 2e-4
LR_META      = 1e-3
EPOCHS       = 50
PATIENCE     = 10
FOCAL_GAMMA  = 2.0
MIXUP_ALPHA  = 0.2
CLASSES      = ['NORMAL', 'PNEUMONIA']
MEAN = [0.5020]; STD = [0.2703]

CLINICAL_FEATURES = [
    'age', 'gender_M', 'is_deceased',
    'heart_rate', 'respiratory_rate', 'spo2', 'systolic_bp', 'diastolic_bp', 'temperature_f',
    'wbc', 'hemoglobin', 'hematocrit', 'creatinine', 'crp', 'alk_phos', 'albumin'
]
N_CLINICAL_FEATURES = len(CLINICAL_FEATURES)

print(f'Image CSV      : {os.path.exists(TEXT_CSV)}')
print(f'Clinical CSV   : {os.path.exists(CLINICAL_CSV)}')
print(f'P1 checkpoint  : {os.path.exists(P1_CKPT)}')
print(f'Save dir       : {SAVE_DIR}')
print(f'Clinical feats : {N_CLINICAL_FEATURES}')

In [ ]:
# --- CELL 2: Data Loading & Merging ---

def resolve_path(p):
    if pd.isna(p) or str(p).strip() == '': return ''
    p = str(p).replace('/', os.sep).replace('\\\\', os.sep)
    return p if os.path.isabs(p) else os.path.join(BASE_DIR, p)

# Load image/label CSV
img_df = pd.read_csv(TEXT_CSV)
img_df['image_path'] = img_df['image_path'].apply(resolve_path)

# Load clinical data
clinical_df = pd.read_csv(CLINICAL_CSV)

# Create gender_M binary feature
if 'gender' in clinical_df.columns:
    clinical_df['gender_M'] = (clinical_df['gender'].str.upper() == 'M').astype(int)

# Merge on study_id
avail_feats = [c for c in CLINICAL_FEATURES if c in clinical_df.columns]
df = img_df.merge(
    clinical_df[['study_id'] + avail_feats],
    on='study_id',
    how='inner'
)

# Update feature list to available only
CLINICAL_FEATURES = avail_feats
N_CLINICAL_FEATURES = len(CLINICAL_FEATURES)

print(f'Image CSV rows    : {len(img_df)}')
print(f'Clinical CSV rows : {len(clinical_df)}')
print(f'After merge       : {len(df)}')
print(f'Using {N_CLINICAL_FEATURES} clinical features: {CLINICAL_FEATURES}')
print(f'\nLabel distribution:')
print(df['label'].value_counts().rename({0: 'Normal', 1: 'Pneumonia'}))

# Fill NaN with median
for col in CLINICAL_FEATURES:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

print(f'\nMissing values after fill: {df[CLINICAL_FEATURES].isnull().sum().sum()}')

In [ ]:
# --- CELL 3: Train / Val / Test Split + StandardScaler ---
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_frac, stratify=train_val_df['label'], random_state=SEED)

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n0 = (split['label']==0).sum(); n1 = (split['label']==1).sum()
    print(f'{name:5s}: {len(split):5d}  (Normal={n0}, Pneumonia={n1})')

# Fit ONLY on training data (no leakage)
scaler = StandardScaler()
train_df = train_df.copy(); val_df = val_df.copy(); test_df = test_df.copy()

train_df[CLINICAL_FEATURES] = scaler.fit_transform(train_df[CLINICAL_FEATURES].astype(float))
val_df[CLINICAL_FEATURES]   = scaler.transform(val_df[CLINICAL_FEATURES].astype(float))
test_df[CLINICAL_FEATURES]  = scaler.transform(test_df[CLINICAL_FEATURES].astype(float))
print('\nClinical features standardised (no data leakage).')

In [ ]:
# --- CELL 4: Bbox Lookup & Dual Dataset Class ---
bbox_df     = pd.read_csv(BBOX_CSV)
bbox_lookup = bbox_df.set_index('image_path').to_dict('index')
print(f'Bbox entries: {len(bbox_lookup)}')

class ImageClinicalDataset(Dataset):
    """Dataset returning (image, clinical_features, label) — no text."""
    def __init__(self, df, bbox_lookup, img_transform, clinical_features):
        self.df               = df.reset_index(drop=True)
        self.bbox_lookup      = bbox_lookup
        self.transform        = img_transform
        self.clinical_features = clinical_features
        self.clahe            = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # --- IMAGE ---
        img = cv2.imread(row.image_path, cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((224,224), dtype=np.uint8)
        img = self.clahe.apply(img)
        bb  = self.bbox_lookup.get(row.image_path, None)
        if bb and bb.get('x_max', 0) > 0:
            img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
        img = self.transform(Image.fromarray(img))

        # --- CLINICAL METADATA ---
        meta = torch.tensor(
            [float(row[c]) for c in self.clinical_features],
            dtype=torch.float32
        )

        return img, meta, int(row.label)

val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
print('Dataset class ready.')

In [ ]:
# --- CELL 5: Model Architecture ---

# ---- CBAM (same as Phase 1, 2, 3) ----
class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.avg=nn.AdaptiveAvgPool2d(1); self.max=nn.AdaptiveMaxPool2d(1)
        self.mlp=nn.Sequential(nn.Conv2d(c,c//r,1,bias=False),nn.ReLU(True),nn.Conv2d(c//r,c,1,bias=False))
        self.sig=nn.Sigmoid()
    def forward(self,x): return x*self.sig(self.mlp(self.avg(x))+self.mlp(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self,k=7):
        super().__init__()
        self.conv=nn.Conv2d(2,1,k,padding=k//2,bias=False); self.sig=nn.Sigmoid()
    def forward(self,x):
        return x*self.sig(self.conv(torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True)[0]],1)))

class CBAM(nn.Module):
    def __init__(self,c,r=16):
        super().__init__()
        self.ca=ChannelAttention(c,r); self.sa=SpatialAttention()
    def forward(self,x): return self.sa(self.ca(x))

# ---- Image Encoder (Phase-1 frozen) ----
class ImageEncoder(nn.Module):
    def __init__(self, ckpt):
        super().__init__()
        xrv_m=xrv.models.DenseNet(weights='densenet121-res224-all')
        self.features=xrv_m.features; self.cbam=CBAM(1024); self.pool=nn.AdaptiveAvgPool2d(1)
        sd={k:v for k,v in torch.load(ckpt, map_location='cpu').items()
            if k.startswith('features.') or k.startswith('cbam.')}
        miss,unexp=self.load_state_dict(sd, strict=False)
        print(f'[ImageEncoder] loaded {len(sd)} keys | missing={len(miss)} unexpected={len(unexp)}')
        for p in self.parameters(): p.requires_grad=False
    def forward(self, x):
        f=F.relu(self.features(x), True); f=self.cbam(f)
        return self.pool(f).flatten(1)   # (B, 1024)

# ---- Metadata Encoder MLP (LayerNorm — works with any batch size including 1) ----
class MetadataEncoder(nn.Module):
    """MLP: N_CLINICAL_FEATURES → 128 → 128 → 64-d.
    Uses LayerNorm instead of BatchNorm1d to avoid crash when batch_size=1."""
    def __init__(self, in_dim, hidden=META_HIDDEN, out_dim=META_OUT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, out_dim),
            nn.ReLU()
        )
        trainable = sum(p.numel() for p in self.net.parameters())
        print(f'[MetadataEncoder] Trainable params: {trainable:,}')
    def forward(self, x):
        return self.net(x)  # (B, 64)

# ---- Image + Clinical Fusion Net ----
class ImageClinicalFusionNet(nn.Module):
    """
    Fuses img_feat (B, 1024) and meta_feat (B, 64).
    Concat -> 1088-d -> MLP -> 2 logits.
    """
    def __init__(self, img_dim=IMG_FEAT_DIM, meta_dim=META_OUT_DIM):
        super().__init__()
        fused_dim = img_dim + meta_dim   # 1088
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(0.4),
            nn.Linear(fused_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
        trainable = sum(p.numel() for p in self.classifier.parameters())
        print(f'[FusionNet] fused_dim={fused_dim} | trainable={trainable:,}')

    def forward(self, img_feat, meta_feat):
        fused = torch.cat([img_feat, meta_feat], dim=1)  # (B, 1088)
        return self.classifier(fused)

print('All model classes defined.')

In [ ]:
# --- CELL 6: Focal Loss & Dual Mixup Utilities ---

class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, weight=None):
        super().__init__()
        self.gamma=gamma; self.weight=weight
    def forward(self, logits, labels):
        ce  = F.cross_entropy(logits, labels, weight=self.weight, reduction='none')
        pt  = torch.exp(-ce)
        return ((1-pt)**self.gamma * ce).mean()

def mixup_dual(img_feat, meta_feat, labels, alpha=MIXUP_ALPHA):
    """Mixup on image and clinical feature embeddings simultaneously."""
    if alpha <= 0: return img_feat, meta_feat, labels, labels, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(img_feat.size(0), device=img_feat.device)
    img_mix  = lam * img_feat  + (1-lam) * img_feat[idx]
    meta_mix = lam * meta_feat + (1-lam) * meta_feat[idx]
    return img_mix, meta_mix, labels, labels[idx], lam

def mixup_criterion(criterion, logits, labels_a, labels_b, lam):
    return lam * criterion(logits, labels_a) + (1-lam) * criterion(logits, labels_b)

print('Focal Loss and Dual Mixup ready.')

In [ ]:
# --- CELL 7: Instantiate Models & DataLoaders ---
print('Loading Phase-1 Image Encoder (Fold 2)...')
image_encoder = ImageEncoder(P1_CKPT).to(DEVICE)

print('\nBuilding MetadataEncoder...')
meta_encoder  = MetadataEncoder(in_dim=N_CLINICAL_FEATURES).to(DEVICE)

print('\nBuilding ImageClinicalFusionNet...')
fusion_model  = ImageClinicalFusionNet().to(DEVICE)

trainable_total = (
    sum(p.numel() for p in fusion_model.parameters() if p.requires_grad) +
    sum(p.numel() for p in meta_encoder.parameters() if p.requires_grad)
)
print(f'\nTotal trainable params: {trainable_total:,}')
print('(Image encoder is fully frozen)')

# DataLoaders — drop_last=True avoids single-sample last batch issues
train_ds = ImageClinicalDataset(train_df, bbox_lookup, train_tfm, CLINICAL_FEATURES)
val_ds   = ImageClinicalDataset(val_df,   bbox_lookup, val_tfm,   CLINICAL_FEATURES)
test_ds  = ImageClinicalDataset(test_df,  bbox_lookup, val_tfm,   CLINICAL_FEATURES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'\nTrain batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
# --- CELL 8: Training Loop ---

def eval_epoch(fusion, img_enc, meta_enc, loader, criterion, device):
    fusion.eval(); img_enc.eval(); meta_enc.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, meta, labels in loader:
            imgs   = imgs.to(device)
            meta   = meta.to(device)
            labels = labels.to(device)
            img_f   = img_enc(imgs)
            meta_f  = meta_enc(meta)
            logits  = fusion(img_f, meta_f)
            loss    = criterion(logits, labels)
            probs   = F.softmax(logits, dim=1)[:,1].cpu().numpy()
            total_loss += loss.item()*labels.size(0)
            all_probs.extend(probs); all_labels.extend(labels.cpu().tolist())
    acc = accuracy_score(all_labels, [1 if p>=0.5 else 0 for p in all_probs])
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels))>1 else 0.5
    return total_loss/len(all_labels), acc, auc, all_probs, all_labels

def find_optimal_threshold(labels, probs):
    fpr,tpr,thresh = roc_curve(labels, probs)
    return float(thresh[np.argmax(tpr-fpr)])

def find_clinical_threshold(labels, probs, target_sens=0.90):
    fpr,tpr,thresh = roc_curve(labels, probs)
    for t,s in zip(thresh, tpr):
        if s >= target_sens:
            return float(t)
    return float(thresh[np.argmax(tpr-fpr)])

# Class weights from training data
n0=(train_df['label']==0).sum(); n1=(train_df['label']==1).sum()
weight=torch.tensor([n1/(n0+n1), n0/(n0+n1)], dtype=torch.float).to(DEVICE)
criterion_focal = FocalLoss(gamma=FOCAL_GAMMA, weight=weight)

# Two separate LR groups
optimizer = torch.optim.AdamW([
    {'params': fusion_model.parameters(), 'lr': LR_FUSION, 'weight_decay': 1e-4},
    {'params': meta_encoder.parameters(), 'lr': LR_META,   'weight_decay': 1e-2},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_auc, best_state, patience_cnt = 0.0, None, 0
history = {'train_loss':[], 'val_loss':[], 'train_auc':[], 'val_auc':[]}

for epoch in range(1, EPOCHS+1):
    fusion_model.train(); image_encoder.eval(); meta_encoder.train()
    ep_loss, ep_probs, ep_labels = 0.0, [], []

    for imgs, meta, labels in train_loader:
        imgs   = imgs.to(DEVICE)
        meta   = meta.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.no_grad():
            img_f = image_encoder(imgs)
        meta_f = meta_encoder(meta)

        img_m, meta_m, la, lb, lam = mixup_dual(img_f, meta_f, labels)

        optimizer.zero_grad()
        logits = fusion_model(img_m, meta_m)
        loss   = mixup_criterion(criterion_focal, logits, la, lb, lam)
        loss.backward()
        nn.utils.clip_grad_norm_(
            list(fusion_model.parameters()) + list(meta_encoder.parameters()), 1.0
        )
        optimizer.step()

        probs = F.softmax(logits.detach(), dim=1)[:,1].cpu().numpy()
        ep_loss += loss.item()*labels.size(0)
        ep_probs.extend(probs); ep_labels.extend(labels.cpu().tolist())

    scheduler.step()
    tr_loss = ep_loss/len(ep_labels)
    tr_auc  = roc_auc_score(ep_labels, ep_probs) if len(set(ep_labels))>1 else 0.5

    val_loss, val_acc, val_auc, _, _ = eval_epoch(
        fusion_model, image_encoder, meta_encoder, val_loader, criterion_focal, DEVICE)

    history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss)
    history['train_auc'].append(tr_auc);   history['val_auc'].append(val_auc)

    marker = ''
    if val_auc > best_auc:
        best_auc = val_auc
        best_state = copy.deepcopy({
            'fusion': fusion_model.state_dict(),
            'meta':   meta_encoder.state_dict()
        })
        torch.save(best_state, os.path.join(SAVE_DIR, 'best_img_clinical_model.pth'))
        patience_cnt=0; marker=' <-- best'
    else:
        patience_cnt+=1

    if epoch%5==0 or marker:
        print(f'Ep {epoch:03d} | tr_loss={tr_loss:.4f} tr_auc={tr_auc:.4f} '
              f'| val_auc={val_auc:.4f} val_acc={val_acc:.3f}{marker}')
    if patience_cnt >= PATIENCE:
        print(f'Early stop at epoch {epoch}.'); break

print(f'\nBest Val AUC: {best_auc:.4f}')

In [ ]:
# --- CELL 9: Test Set Evaluation ---
fusion_model.load_state_dict(best_state['fusion'])
meta_encoder.load_state_dict(best_state['meta'])

_,test_acc,test_auc,test_probs,test_labels = eval_epoch(
    fusion_model, image_encoder, meta_encoder, test_loader, criterion_focal, DEVICE)

thresh_youden   = find_optimal_threshold(test_labels, test_probs)
thresh_clinical = find_clinical_threshold(test_labels, test_probs, target_sens=0.90)

preds_def      = [1 if p>=0.5            else 0 for p in test_probs]
preds_youden   = [1 if p>=thresh_youden  else 0 for p in test_probs]
preds_clinical = [1 if p>=thresh_clinical else 0 for p in test_probs]

def metrics(labels, preds, name, thresh):
    cm = confusion_matrix(labels, preds)
    s  = cm[1,1]/(cm[1,0]+cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0
    sp = cm[0,0]/(cm[0,0]+cm[0,1]) if (cm[0,0]+cm[0,1])>0 else 0
    ac = (cm[0,0]+cm[1,1])/cm.sum()
    f1 = f1_score(labels, preds)
    print(f'{name} (thresh={thresh:.3f}): Acc={ac:.3f} F1={f1:.3f} Sens={s:.3f} Spec={sp:.3f}')
    return s, sp, ac, f1

print(f'Test AUC: {test_auc:.4f}\n')
s_d,sp_d,ac_d,f1_d = metrics(test_labels, preds_def,      'Default  ', 0.5)
s_y,sp_y,ac_y,f1_y = metrics(test_labels, preds_youden,   'Youden-J ', thresh_youden)
s_c,sp_c,ac_c,f1_c = metrics(test_labels, preds_clinical, 'Clinical ', thresh_clinical)

In [ ]:
# --- CELL 10: Confusion Matrices ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, preds, title in zip(axes,
    [preds_def, preds_youden, preds_clinical],
    ['Default (0.5)', f'Youden-J ({thresh_youden:.3f})', f'Clinical ({thresh_clinical:.3f})']):
    cm_arr = confusion_matrix(test_labels, preds)
    sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASSES, yticklabels=CLASSES)
    ax.set_title(f'Img+Clinical | {title}', fontsize=13)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.suptitle(f'Image + Clinical Fusion | Test AUC={test_auc:.4f}', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'img_clinical_confusion.png'), bbox_inches='tight', dpi=150)
plt.show(); print('Saved confusion matrices.')

In [ ]:
# --- CELL 11: ROC Curve ---
fpr, tpr, _ = roc_curve(test_labels, test_probs)
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, lw=2, color='teal', label=f'Img+Clinical (AUC={test_auc:.4f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Image + Clinical Fusion — ROC Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'img_clinical_roc.png'), dpi=150)
plt.show(); print('Saved ROC curve.')

In [ ]:
# --- CELL 12: Training Curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_auc'], label='Train AUC', color='teal')
axes[0].plot(history['val_auc'],   label='Val AUC',   color='darkorange')
axes[0].axhline(0.820, ls='--', color='gray',  alpha=0.7, label='Phase 1 baseline (0.820)')
axes[0].axhline(0.984, ls='--', color='green', alpha=0.7, label='Phase 3 Full (0.984)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('AUC')
axes[0].set_title('AUC over Epochs'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_loss'], label='Train Loss', color='teal')
axes[1].plot(history['val_loss'],   label='Val Loss',   color='darkorange')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Focal Loss')
axes[1].set_title('Loss over Epochs'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Image + Clinical Fusion — Training History', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'img_clinical_training_curves.png'), dpi=150)
plt.show(); print('Saved training curves.')

In [ ]:
# --- CELL 13: Modality Ablation Comparison Chart ---
phases = [
    {'name': 'Phase 1\nImage Only',             'auc': 0.820, 'sens': 69.7,  'color': '#4477AA'},
    {'name': 'Phase 3b\nImage + Clinical',       'auc': round(test_auc, 4),
                                                 'sens': round(s_y*100, 1),   'color': '#228833'},
    {'name': 'Phase 2v2\nImage + Text',          'auc': 0.949, 'sens': 91.4,  'color': '#66CCEE'},
    {'name': 'Phase 3 Full\nImg+Text+Clinical',  'auc': 0.984, 'sens': 94.9,  'color': '#EE6677'},
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
names  = [p['name'] for p in phases]
colors = [p['color'] for p in phases]

axes[0].bar(names, [p['auc'] for p in phases], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_ylim(0.75, 1.0)
axes[0].set_ylabel('AUC'); axes[0].set_title('Test AUC — Modality Ablation Study')
axes[0].grid(axis='y', alpha=0.3)
for i,p in enumerate(phases):
    axes[0].text(i, p['auc']+0.002, f"{p['auc']:.3f}", ha='center', fontsize=11, fontweight='bold')

axes[1].bar(names, [p['sens'] for p in phases], color=colors, edgecolor='black', alpha=0.85)
axes[1].axhline(90, ls='--', color='red', alpha=0.6, label='90% clinical target')
axes[1].set_ylim(50, 100)
axes[1].set_ylabel('Recall / Sensitivity (%)')
axes[1].set_title('Sensitivity (Recall) — Modality Ablation Study')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
for i,p in enumerate(phases):
    axes[1].text(i, p['sens']+0.5, f"{p['sens']:.1f}%", ha='center', fontsize=11, fontweight='bold')

plt.suptitle('PneumoFusionNet — Modality Ablation Study', fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'modality_ablation.png'), bbox_inches='tight', dpi=150)
plt.show(); print('Saved modality ablation chart.')

In [ ]:
# --- CELL 14: Clinical Feature Importance (Permutation-based) ---
print('Computing clinical feature importances...')
fusion_model.eval(); image_encoder.eval(); meta_encoder.eval()

_,_,baseline_auc,_,_ = eval_epoch(
    fusion_model, image_encoder, meta_encoder, test_loader, criterion_focal, DEVICE)
print(f'Baseline AUC: {baseline_auc:.4f}')

feature_importances = {}
for feat_idx, feat_name in enumerate(CLINICAL_FEATURES):
    all_probs_perm, all_labels_perm = [], []
    with torch.no_grad():
        for imgs, meta, labels in test_loader:
            imgs   = imgs.to(DEVICE)
            meta   = meta.clone().to(DEVICE)
            labels = labels.to(DEVICE)
            perm_idx = torch.randperm(meta.size(0))
            meta[:, feat_idx] = meta[perm_idx, feat_idx]
            img_f  = image_encoder(imgs)
            meta_f = meta_encoder(meta)
            logits = fusion_model(img_f, meta_f)
            probs  = F.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_probs_perm.extend(probs)
            all_labels_perm.extend(labels.cpu().tolist())
    perm_auc = roc_auc_score(all_labels_perm, all_probs_perm)
    feature_importances[feat_name] = baseline_auc - perm_auc

sorted_feats = sorted(feature_importances.items(), key=lambda x: x[1], reverse=True)
feat_names = [f[0] for f in sorted_feats]
feat_imps  = [f[1] for f in sorted_feats]

plt.figure(figsize=(10, 6))
bar_colors = ['#EE6677' if v > 0 else '#4477AA' for v in feat_imps]
plt.barh(feat_names, feat_imps, color=bar_colors, edgecolor='black', alpha=0.85)
plt.axvline(0, color='black', lw=1)
plt.xlabel('AUC Drop when Feature Permuted (higher = more important)')
plt.title('Clinical Feature Importance (Permutation)\nImage + Clinical Fusion')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'img_clinical_feature_importance.png'), dpi=150)
plt.show()
print('\nFeature Importances:')
for name, imp in sorted_feats:
    print(f'  {name:20s}: {imp:+.4f}')

In [ ]:
# --- CELL 15: Save Results JSON ---
results = {
    'model': 'Image + Clinical Metadata Fusion (No Text)',
    'description': 'Phase 1 DenseNet+CBAM (frozen) + Clinical MLP. No radiology text used.',
    'architecture': {
        'image_encoder': 'DenseNet121+CBAM (frozen, Phase 1.1v4 Fold 2) → 1024-d',
        'meta_encoder':  f'MLP+LayerNorm({N_CLINICAL_FEATURES}→128→128→64)',
        'fusion':        'Concat [img(1024) + meta(64)] = 1088-d → MLP classifier',
        'clinical_feats': CLINICAL_FEATURES,
    },
    'training': {
        'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df),
        'batch_size': BATCH_SIZE, 'epochs_run': len(history['val_auc']),
        'lr_fusion': LR_FUSION, 'lr_meta': LR_META,
        'focal_gamma': FOCAL_GAMMA, 'mixup_alpha': MIXUP_ALPHA
    },
    'results': {
        'test_auc':              round(test_auc, 4),
        'best_val_auc':          round(best_auc, 4),
        'default_acc':           round(ac_d, 4),
        'default_sensitivity':   round(s_d, 4),
        'default_specificity':   round(sp_d, 4),
        'youden_threshold':      round(thresh_youden, 4),
        'youden_acc':            round(ac_y, 4),
        'youden_sensitivity':    round(s_y, 4),
        'youden_specificity':    round(sp_y, 4),
        'clinical_threshold':    round(thresh_clinical, 4),
        'clinical_sensitivity':  round(s_c, 4),
        'clinical_specificity':  round(sp_c, 4),
        'n_test': len(test_labels)
    },
    'feature_importances': {k: round(v, 4) for k, v in feature_importances.items()},
    'modality_comparison': {
        'phase1_image_only':          0.820,
        'phase3b_image_clinical':     round(test_auc, 4),
        'phase2v2_image_text':        0.949,
        'phase3_image_text_clinical': 0.984,
        'clinical_value_over_image':  round(test_auc - 0.820, 4),
        'text_value_over_clinical':   round(0.984 - test_auc, 4)
    }
}

out_path = os.path.join(SAVE_DIR, 'image_clinical_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print('='*60)
print('IMAGE + CLINICAL FUSION — FINAL RESULTS')
print('='*60)
print(f'Test AUC         : {test_auc:.4f}')
print(f'Sensitivity (Y-J): {s_y*100:.1f}%')
print(f'Specificity (Y-J): {sp_y*100:.1f}%')
print(f'Accuracy    (Y-J): {ac_y*100:.1f}%')
print(f'\nModality Ablation Summary:')
print(f'  Phase 1  (Image only):            AUC = 0.820')
print(f'  Phase 3b (Image + Clinical):      AUC = {test_auc:.4f}  (+{test_auc-0.820:.4f} vs Phase 1)')
print(f'  Phase 2v2 (Image + Text):         AUC = 0.949')
print(f'  Phase 3  (Img + Text + Clinical): AUC = 0.984')
print(f'\nResults saved to: {out_path}')